# Entrenar GPT desde cero — Colab (v2: ~350M params)

Notebook autocontenido para entrenar un modelo GPT (~350M params) en Google Colab con GPU.

**Cambios respecto a v1 (38M):**
- Arquitectura escalada: 1024 embd / 16 heads / 24 layers / 1024 ctx
- Múltiples fuentes de datos: Wikipedia + CulturaX (objetivo 8–10B tokens)
- BF16 mixed precision (nativo en Blackwell, sin GradScaler)
- torch.compile para optimización automática
- DataLoader con stride = block_size + memory-mapping
- Gradient accumulation para batch efectivo de 128
- Resume automático desde checkpoint

**Flujo:**
1. Montar Google Drive y configurar rutas
2. Instalar dependencias
3. Construir corpus (Wikipedia + CulturaX) o cargar existente
4. Tokenizar con tiktoken + vocabulario compacto
5. Entrenar
6. Generar texto

## 1. Setup: Drive + dependencias

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Ruta del proyecto en Drive
PROJECT_DIR = "/content/drive/MyDrive/llm-desde-cero"

import os
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/checkpoints", exist_ok=True)

print(f"Proyecto en: {PROJECT_DIR}")
print(f"Contenido: {os.listdir(PROJECT_DIR)}")

In [ ]:
!pip install tiktoken datasets tqdm -q

In [ ]:
# Verificar GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"BF16 soportado: {torch.cuda.is_bf16_supported()}")
    print(f"Compute capability: {torch.cuda.get_device_capability()}")

## 2. Construir corpus

**Objetivo: ~8–10B tokens** para 350M parámetros (Chinchilla: ~20 tokens/param).

- Si ya tienes los archivos de corpus en Drive, las celdas de descarga se saltan automáticamente.
- Cada fuente se tokeniza y guarda por separado para poder ajustar la mezcla sin re-tokenizar.

### 2a. Wikipedia en español (~570M tokens)

In [ ]:
import re
from datasets import load_dataset

CORPUS_DIR = f"{PROJECT_DIR}/data"
WIKI_CORPUS_PATH = f"{CORPUS_DIR}/corpus_wiki.txt"

TARGET_CHARS_WIKI = 2_000_000_000
MIN_ARTICLE_LEN   = 500
MIN_LINE_LEN      = 70
MIN_PROSE_RATIO   = 0.30

SECTIONS_TO_REMOVE = {
    "referencias", "notas", "notas y referencias", "notas al pie",
    "fuentes", "bibliografía", "fuentes y bibliografía",
    "véase también", "enlaces externos", "links externos",
    "discografía", "filmografía", "videografía", "ludografía",
    "obras", "obras principales", "obras destacadas", "obras publicadas",
    "publicaciones", "libros", "álbumes", "sencillos", "singles",
    "premios", "premios y reconocimientos", "premios y nominaciones",
    "palmarés", "distinciones", "honores", "condecoraciones", "medallas",
    "estadísticas", "datos estadísticos", "resultados", "temporadas",
    "carrera deportiva", "clubes", "trayectoria deportiva",
    "galería", "galería de imágenes", "galería fotográfica",
    "cronología", "genealogía", "heráldica", "escudo",
}


def remove_sections(text: str) -> str:
    lines = text.split('\n')
    result, skip, skip_level = [], False, 0

    for line in lines:
        m = re.match(r'^(={2,})\s*(.+?)\s*\1\s*$', line)
        if m:
            level = len(m.group(1))
            title = m.group(2).strip().lower()
            if title in SECTIONS_TO_REMOVE:
                skip, skip_level = True, level
                continue
            if skip and level <= skip_level:
                skip = False
        if not skip:
            result.append(line)

    return '\n'.join(result)


def remove_wiki_markup(text: str) -> str:
    for _ in range(6):
        prev = text
        text = re.sub(r'\{\{[^{}]*\}\}', '', text, flags=re.DOTALL)
        if text == prev:
            break
    text = re.sub(r'\{\|.*?\|\}', '', text, flags=re.DOTALL)
    text = re.sub(r'\[\[Categoría:[^\]]*\]\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\[Archivo:[^\]]*\]\]',   '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\[File:[^\]]*\]\]',      '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\[Imagen:[^\]]*\]\]',    '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\[(?:[^\]|]*\|)?([^\]|]+)\]\]', r'\1', text)
    text = re.sub(r"'{2,3}", '', text)
    text = re.sub(r'^={2,}\s*(.+?)\s*={2,}\s*$', r'\1', text, flags=re.MULTILINE)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'https?://\S+', '', text)
    return text


def filter_prose_lines(text: str):
    lines = text.split('\n')
    result, chars_in, chars_out = [], len(text), 0

    for line in lines:
        stripped = line.strip()
        if not stripped:
            result.append('')
            continue
        if stripped[0] in ('*', '-', '#', '|', '!'):
            continue
        if re.match(r'^\d+\.', stripped):
            continue
        if re.match(r'^\d{3,4}:', stripped):
            continue
        if stripped.count('|') >= 2:
            continue
        if len(stripped) < MIN_LINE_LEN:
            continue
        if sum(c.isalpha() for c in stripped) / len(stripped) < 0.50:
            continue
        result.append(line)
        chars_out += len(line)

    prose_ratio = chars_out / chars_in if chars_in > 0 else 0.0
    return '\n'.join(result), prose_ratio


def clean_article(text: str):
    text = remove_sections(text)
    text = remove_wiki_markup(text)
    text, prose_ratio = filter_prose_lines(text)
    if prose_ratio < MIN_PROSE_RATIO:
        return None
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    if len(text) < MIN_ARTICLE_LEN:
        return None
    return text


# --- Construir o cargar corpus Wikipedia ---
if os.path.exists(WIKI_CORPUS_PATH):
    corpus_size = os.path.getsize(WIKI_CORPUS_PATH)
    print(f"[Wikipedia] Ya existe: {corpus_size / 1e9:.2f} GB "
          f"(~{corpus_size / 3.5 / 1e6:.0f}M tokens est.)")
else:
    print(f"[Wikipedia] Objetivo: {TARGET_CHARS_WIKI / 1e9:.1f}B chars")
    print("[Wikipedia] Descargando (streaming)...\n")

    dataset = load_dataset("wikimedia/wikipedia", "20231101.es",
                           split="train", streaming=True)

    textos, total_chars, n_articles, n_skipped = [], 0, 0, 0

    for article in dataset:
        raw_text = article["text"]
        if len(raw_text) < MIN_ARTICLE_LEN * 2:
            n_skipped += 1
            continue
        text = clean_article(raw_text)
        if text is None:
            n_skipped += 1
            continue

        textos.append(text)
        total_chars += len(text)
        n_articles += 1

        if n_articles % 5000 == 0:
            pct = 100 * total_chars / TARGET_CHARS_WIKI
            print(f"  {n_articles:,} art | {total_chars / 1e6:.0f}M chars ({pct:.1f}%) | skip: {n_skipped:,}")
        if total_chars >= TARGET_CHARS_WIKI:
            break

    corpus = "\n\n".join(textos)
    with open(WIKI_CORPUS_PATH, 'w', encoding='utf-8') as f:
        f.write(corpus)

    print(f"\n[✓] Wikipedia: {n_articles:,} art | {len(corpus)/1e6:.0f}M chars")
    del corpus, textos

### 2b. CulturaX — texto web español (~7B tokens)

CulturaX es un corpus web limpio y deduplicado. Fuente principal de volumen.

**Esta descarga puede tardar varias horas.** Se guarda en Drive progresivamente.

In [ ]:
CULTURAX_CORPUS_PATH = f"{CORPUS_DIR}/corpus_culturax.txt"
CULTURAX_PROGRESS    = f"{CORPUS_DIR}/corpus_culturax.progress"
TARGET_CHARS_CULTURAX = 24_500_000_000  # ~7B tokens
import json


def clean_web_text(text: str) -> str | None:
    if len(text) < 200:
        return None
    alpha_ratio = sum(c.isalpha() for c in text) / len(text)
    if alpha_ratio < 0.60:
        return None
    lines = text.strip().split('\n')
    unique_lines = set(lines)
    if len(unique_lines) / max(len(lines), 1) < 0.5:
        return None
    text = '\n'.join(l for l in lines if len(l) < 1000)
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    if len(text) < 200:
        return None
    return text


# --- Cargar progreso si existe ---
progress = {"n_docs": 0, "n_skipped": 0, "total_chars": 0, "n_seen": 0}
if os.path.exists(CULTURAX_PROGRESS):
    with open(CULTURAX_PROGRESS) as f:
        progress = json.load(f)

if progress["total_chars"] >= TARGET_CHARS_CULTURAX * 0.95:
    print(f"[CulturaX] Completo: {progress['total_chars']/1e9:.1f}B chars "
          f"({progress['n_docs']:,} docs)")
else:
    if progress["n_seen"] > 0:
        print(f"[CulturaX] Descarga parcial: {progress['total_chars']/1e9:.1f}B chars")
        print(f"[CulturaX] Resumiendo desde doc {progress['n_seen']:,} "
              f"(puede tardar unos minutos en avanzar)...")
    else:
        print(f"[CulturaX] Objetivo: {TARGET_CHARS_CULTURAX / 1e9:.1f}B chars")
        print("[CulturaX] Descargando (streaming)...")

    dataset = load_dataset("uonlp/CulturaX", "es", split="train", streaming=True)

    # Saltar docs ya procesados
    if progress["n_seen"] > 0:
        dataset = dataset.skip(progress["n_seen"])

    total_chars = progress["total_chars"]
    n_docs      = progress["n_docs"]
    n_skipped   = progress["n_skipped"]
    n_seen      = progress["n_seen"]

    # Abrir en modo append si hay progreso previo
    mode = 'a' if progress["n_seen"] > 0 else 'w'

    with open(CULTURAX_CORPUS_PATH, mode, encoding='utf-8') as f:
        for doc in dataset:
            n_seen += 1
            text = clean_web_text(doc["text"])
            if text is None:
                n_skipped += 1
                continue

            f.write(text + "\n\n")
            total_chars += len(text)
            n_docs += 1

            # Guardar progreso cada 10k docs (para poder resumir)
            if n_docs % 10000 == 0:
                f.flush()
                with open(CULTURAX_PROGRESS, 'w') as pf:
                    json.dump({"n_docs": n_docs, "n_skipped": n_skipped,
                               "total_chars": total_chars, "n_seen": n_seen}, pf)

            if n_docs % 50000 == 0:
                pct = 100 * total_chars / TARGET_CHARS_CULTURAX
                print(f"  {n_docs:,} docs | {total_chars/1e9:.1f}B chars ({pct:.1f}%) | skip: {n_skipped:,}")

            if total_chars >= TARGET_CHARS_CULTURAX:
                break

    # Progreso final
    with open(CULTURAX_PROGRESS, 'w') as pf:
        json.dump({"n_docs": n_docs, "n_skipped": n_skipped,
                   "total_chars": total_chars, "n_seen": n_seen}, pf)

    print(f"\n[✓] CulturaX: {n_docs:,} docs | {total_chars/1e9:.1f}B chars")

## 3. Tokenizador (tiktoken + vocabulario compacto)

In [ ]:
import json
import tiktoken
import numpy as np


class TiktokenWrapper:

    def __init__(self, encoding_name: str = "cl100k_base"):
        self.encoding_name = encoding_name
        self.enc = tiktoken.get_encoding(encoding_name)
        self.tiktoken_to_compact: dict[int, int] = {}
        self.compact_to_tiktoken: dict[int, int] = {}
        self.vocab_size: int = 0

    def build_vocab_from_files(self, corpus_paths: list[str], token_out_paths: list[str],
                               chunk_chars: int = 100_000_000):
        """
        Tokeniza múltiples corpus en chunks de ~100MB (bajo consumo de RAM).
        Pass 1: tokeniza por chunks → escribe IDs raw de tiktoken a .raw
        Pass 2: construye vocab compacto, remapea con lookup vectorizado → .bin final
        """
        all_unique = set()
        raw_paths = []
        token_counts = []

        # Pass 1: tokenizar por chunks y escribir IDs raw
        for path in corpus_paths:
            name = os.path.basename(path)
            raw_path = os.path.join('/tmp', os.path.basename(path) + '.tokens.raw')
            raw_paths.append(raw_path)

            n_tokens = 0
            print(f'[Tokenizer] Pass 1 — tokenizando {name}...')
            with open(path, 'r', encoding='utf-8') as fin, \
                 open(raw_path, 'wb') as fout:
                while True:
                    chunk = fin.read(chunk_chars)
                    if not chunk:
                        break
                    tids = self.enc.encode(chunk)
                    all_unique.update(tids)
                    np.array(tids, dtype=np.int32).tofile(fout)
                    n_tokens += len(tids)
                    print(f'  {n_tokens/1e6:.0f}M tokens...', end='\r')

            token_counts.append(n_tokens)
            print(f'  {name}: {n_tokens:,} tokens          ')

        # Construir vocab compacto
        unique_ids = sorted(all_unique)
        self.tiktoken_to_compact = {orig: compact for compact, orig in enumerate(unique_ids)}
        self.compact_to_tiktoken = {compact: orig for orig, compact in self.tiktoken_to_compact.items()}
        self.vocab_size = len(unique_ids)
        print(f'[Tokenizer] Vocab compacto: {self.vocab_size:,} tokens '
              f'(de {self.enc.n_vocab:,})')

        # Lookup table vectorizado para remapeo rápido
        max_tid = max(self.tiktoken_to_compact.keys()) + 1
        lookup = np.zeros(max_tid + 1, dtype=np.int32)
        for orig, compact in self.tiktoken_to_compact.items():
            lookup[orig] = compact

        # Pass 2: remapear raw → compact en chunks
        total_tokens = 0
        remap_chunk = 10_000_000  # 10M tokens a la vez (~40MB RAM)

        for raw_path, out_path, n_tokens in zip(raw_paths, token_out_paths, token_counts):
            print(f'[Tokenizer] Pass 2 — remapeando → {os.path.basename(out_path)}...')
            raw = np.memmap(raw_path, dtype=np.int32, mode='r', shape=(n_tokens,))
            out = np.memmap(out_path, dtype=np.int32, mode='w+', shape=(n_tokens,))

            for i in range(0, n_tokens, remap_chunk):
                end = min(i + remap_chunk, n_tokens)
                out[i:end] = lookup[np.array(raw[i:end])]

            out.flush()
            del raw, out
            os.remove(raw_path)  # borrar archivo temporal
            total_tokens += n_tokens
            print(f'  {os.path.basename(out_path)}: {n_tokens:,} tokens')

        del lookup
        print(f'[Tokenizer] Total: {total_tokens:,} tokens')
        return total_tokens

    def encode(self, text: str) -> list[int]:
        tids = self.enc.encode(text)
        return [self.tiktoken_to_compact[t] for t in tids if t in self.tiktoken_to_compact]

    def decode(self, compact_ids: list[int]) -> str:
        tids = [self.compact_to_tiktoken[c] for c in compact_ids]
        return self.enc.decode(tids)

    def save(self, path: str):
        data = {
            "encoding_name": self.encoding_name,
            "vocab_size": self.vocab_size,
            "tiktoken_to_compact": {str(k): v for k, v in self.tiktoken_to_compact.items()},
            "compact_to_tiktoken": {str(k): v for k, v in self.compact_to_tiktoken.items()},
        }
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f)
        print(f"[Tokenizer] Vocab guardado en {path}")

    def load(self, path: str):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.encoding_name = data["encoding_name"]
        self.enc = tiktoken.get_encoding(self.encoding_name)
        self.vocab_size = data["vocab_size"]
        self.tiktoken_to_compact = {int(k): v for k, v in data["tiktoken_to_compact"].items()}
        self.compact_to_tiktoken = {int(k): v for k, v in data["compact_to_tiktoken"].items()}
        print(f"[Tokenizer] Vocab cargado ({self.vocab_size:,} tokens)")

### 3b. Tokenizar y concatenar

Los `.bin` tokenizados se guardan en **Drive** (persisten entre sesiones). El archivo concatenado `all_tokens.bin` se genera en **disco local** (`/content/`) para que el DataLoader lea rápido. Se regenera automáticamente al inicio de cada sesión (~1-2 min).

La tokenización se hace en dos pasadas con bajo consumo de RAM: primero tokeniza por chunks de ~100MB y recoge los IDs únicos, luego remapea a IDs compactos con una lookup table vectorizada.

In [ ]:
# .bin de tokens en Drive (persisten entre sesiones)
TOKENS_WIKI_PATH     = f"{CORPUS_DIR}/tokens_wiki.bin"
TOKENS_CULTURAX_PATH = f"{CORPUS_DIR}/tokens_culturax.bin"
VOCAB_PATH           = f"{CORPUS_DIR}/vocab.json"

# all_tokens.bin en disco LOCAL (rápido para I/O durante training)
# Se regenera al inicio de cada sesión copiando desde los .bin de Drive
ALL_TOKENS_PATH = "/content/all_tokens.bin"

tokenizer = TiktokenWrapper()

# Determinar qué corpus hay disponibles
corpus_paths, token_paths = [], []
if os.path.exists(WIKI_CORPUS_PATH):
    corpus_paths.append(WIKI_CORPUS_PATH)
    token_paths.append(TOKENS_WIKI_PATH)
if os.path.exists(CULTURAX_CORPUS_PATH):
    corpus_paths.append(CULTURAX_CORPUS_PATH)
    token_paths.append(TOKENS_CULTURAX_PATH)

assert len(corpus_paths) > 0, "No se encontró ningún corpus."

# Tokenizar si faltan .bin (resultado va a Drive)
needs_tokenize = not all(os.path.exists(p) for p in token_paths)

if needs_tokenize:
    total_tokens = tokenizer.build_vocab_from_files(corpus_paths, token_paths)
    tokenizer.save(VOCAB_PATH)
else:
    tokenizer.load(VOCAB_PATH)
    total_tokens = sum(
        os.path.getsize(p) // 4  # int32 = 4 bytes
        for p in token_paths
    )
    print(f"[Tokenizer] Tokens ya procesados: {total_tokens:,}")

# Concatenar en disco LOCAL (se rehace cada sesión, tarda ~1-2 min)
if not os.path.exists(ALL_TOKENS_PATH):
    print(f"[Data] Concatenando .bin de Drive → disco local...")
    all_tokens = np.memmap(ALL_TOKENS_PATH, dtype=np.int32, mode='w+', shape=(total_tokens,))
    offset = 0
    for p in token_paths:
        n = os.path.getsize(p) // 4
        chunk = np.memmap(p, dtype=np.int32, mode='r', shape=(n,))
        all_tokens[offset:offset + n] = chunk
        offset += n
        del chunk
        print(f"  {os.path.basename(p)}: {n:,} tokens")
    all_tokens.flush()
    del all_tokens
    print(f"[Data] Listo en disco local: {ALL_TOKENS_PATH}")
else:
    print(f"[Data] Archivo local ya existe")

print(f"\n[Resumen] Tokens: {total_tokens:,} | Vocab: {tokenizer.vocab_size:,}")
print(f"  .bin en Drive (persistente): {CORPUS_DIR}/")
print(f"  all_tokens.bin en local (rápido): {ALL_TOKENS_PATH}")

## 4. Dataset y DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader


class TextDataset(Dataset):
    """
    Dataset con stride = block_size (ventanas no solapantes) y memory-mapping.
    No carga todos los tokens en RAM.
    """

    def __init__(self, tokens_path: str, block_size: int,
                 start_idx: int = 0, end_idx: int | None = None):
        self.tokens = np.memmap(tokens_path, dtype=np.int32, mode='r')
        self.start_idx = start_idx
        self.end_idx = end_idx or len(self.tokens)
        self.block_size = block_size
        usable = self.end_idx - self.start_idx - 1
        self.n_examples = usable // block_size

    def __len__(self):
        return self.n_examples

    def __getitem__(self, idx):
        start = self.start_idx + idx * self.block_size
        chunk = np.array(self.tokens[start : start + self.block_size + 1])
        x = torch.from_numpy(chunk[:-1].astype(np.int64))
        y = torch.from_numpy(chunk[1:].astype(np.int64))
        return x, y


def build_dataloaders(tokens_path, total_tokens, block_size, batch_size,
                      val_split=0.1, num_workers=4):
    split_idx = int(total_tokens * (1 - val_split))
    train_ds = TextDataset(tokens_path, block_size, start_idx=0, end_idx=split_idx)
    val_ds   = TextDataset(tokens_path, block_size, start_idx=split_idx)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)

    print(f"[Data] Train: {len(train_ds):,} ex | Val: {len(val_ds):,} ex")
    print(f"[Data] Batch: {batch_size} | Block: {block_size}")
    return train_loader, val_loader

## 5. Arquitectura GPT

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


class MultiHeadSelfAttention(nn.Module):

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.n_embd = n_head, n_embd
        self.d_k = n_embd // n_head
        self.dropout_p = dropout
        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        Q, K, V = qkv.split(self.n_embd, dim=2)
        Q = Q.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        out = F.scaled_dot_product_attention(
            Q, K, V, dropout_p=self.dropout_p if self.training else 0.0, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_embd)
        return self.resid_dropout(self.out_proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):

    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.transformer = nn.ModuleDict({
            'tok_emb': nn.Embedding(vocab_size, n_embd),
            'pos_emb': nn.Embedding(block_size, n_embd),
            'drop':    nn.Dropout(dropout),
            'blocks':  nn.ModuleList([
                TransformerBlock(n_embd, n_head, block_size, dropout)
                for _ in range(n_layer)]),
            'ln_f':    nn.LayerNorm(n_embd),
        })
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.transformer['tok_emb'].weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.transformer['drop'](
            self.transformer['tok_emb'](idx) + self.transformer['pos_emb'](pos))
        for block in self.transformer['blocks']:
            x = checkpoint(block, x, use_reentrant=False)
        x = self.transformer['ln_f'](x)

        if targets is not None:
            # Chunked cross-entropy: nunca existe el tensor (B, T, vocab_size) completo.
            # Procesamos chunk_size tokens a la vez → máximo (B, chunk_size, vocab_size) en memoria.
            chunk_size = 128
            loss = torch.tensor(0.0, device=idx.device)
            for i in range(0, T, chunk_size):
                x_chunk      = x[:, i:i+chunk_size, :]
                t_chunk      = targets[:, i:i+chunk_size]
                logits_chunk = self.lm_head(x_chunk)
                loss += F.cross_entropy(
                    logits_chunk.view(-1, logits_chunk.size(-1)),
                    t_chunk.contiguous().view(-1)
                ) * (x_chunk.size(1) / T)  # promedio ponderado por tamaño real del chunk
            logits = None  # no se materializa durante training
        else:
            # Inferencia: T=1, el tensor es trivial
            logits = self.lm_head(x)
            loss = None

        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())


print("Arquitectura GPT definida.")

## 6. Configuración y entrenamiento

In [ ]:
import math
import time
from torch.amp import autocast

# --- Hiperparámetros (~350M params) ---
config = {
    # Modelo
    "n_embd"      : 1024,
    "n_head"      : 16,
    "n_layer"     : 24,
    "block_size"  : 1024,
    "dropout"     : 0.1,

    # Entrenamiento
    "batch_size"  : 64,
    "grad_accum"  : 2,         # batch efectivo = 64 × 2 = 128
    "max_iters"   : 75000,
    "eval_interval": 500,
    "eval_iters"  : 200,

    # Optimizador
    "learning_rate": 3e-4,
    "weight_decay" : 0.1,
    "beta1"        : 0.9,
    "beta2"        : 0.95,
    "grad_clip"    : 1.0,

    # LR scheduler
    "warmup_iters"  : 2000,
    "lr_decay_iters": 75000,
    "min_lr"        : 3e-5,

    # Checkpointing
    "checkpoint_interval": 1000,
    "checkpoint_dir": f"{PROJECT_DIR}/checkpoints",

    # Rutas
    "tokens_path" : ALL_TOKENS_PATH,
    "vocab_path"  : VOCAB_PATH,
}

# Info sobre iteraciones
tokens_per_iter = config["batch_size"] * config["grad_accum"] * config["block_size"]
iters_per_epoch = total_tokens // tokens_per_iter

print(f"Tokens por iteración:  {tokens_per_iter:,}")
print(f"Iteraciones por época: {iters_per_epoch:,}")
print(f"max_iters:             {config['max_iters']:,}")
print(f"Épocas aprox:          {config['max_iters'] / iters_per_epoch:.2f}")

if config["max_iters"] > iters_per_epoch * 1.5:
    print(f"\n⚠️  Estás en multi-epoch. Considera más datos o menos iteraciones.")

print("\nConfiguración:")
for k, v in config.items():
    print(f"  {k}: {v}")

In [ ]:
def get_lr(it):
    lr, min_lr = config["learning_rate"], config["min_lr"]
    if it < config["warmup_iters"]:
        return lr * it / config["warmup_iters"]
    if it > config["lr_decay_iters"]:
        return min_lr
    decay_ratio = (it - config["warmup_iters"]) / (config["lr_decay_iters"] - config["warmup_iters"])
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (lr - min_lr)


@torch.no_grad()
def evaluate(model, val_loader, device):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_loader):
        if i >= config["eval_iters"]:
            break
        x, y = x.to(device), y.to(device)
        with autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[Train] Dispositivo: {device}")
    os.makedirs(config["checkpoint_dir"], exist_ok=True)

    # --- DataLoaders ---
    train_loader, val_loader = build_dataloaders(
        tokens_path=config["tokens_path"], total_tokens=total_tokens,
        block_size=config["block_size"], batch_size=config["batch_size"])

    # --- Modelo ---
    model = GPT(
        vocab_size=tokenizer.vocab_size, n_embd=config["n_embd"],
        n_head=config["n_head"], n_layer=config["n_layer"],
        block_size=config["block_size"], dropout=config["dropout"],
    ).to(device)
    config["vocab_size"] = tokenizer.vocab_size
    print(f"[Train] Parámetros: {model.num_params():,}")

    # Compilar para mayor velocidad
    print("[Train] Compilando modelo con torch.compile...")
    model = torch.compile(model)

    # --- Optimizador ---
    decay_params   = [p for n, p in model.named_parameters() if p.dim() >= 2]
    nodecay_params = [p for n, p in model.named_parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW([
        {"params": decay_params,   "weight_decay": config["weight_decay"]},
        {"params": nodecay_params, "weight_decay": 0.0},
    ], lr=config["learning_rate"], betas=(config["beta1"], config["beta2"]))

    # --- Resume desde checkpoint ---
    resume_path = os.path.join(config["checkpoint_dir"], "latest.pt")
    start_iter = 1
    best_val_loss = float('inf')

    if os.path.exists(resume_path):
        print(f"[Train] Cargando checkpoint...")
        ckpt = torch.load(resume_path, map_location=device)
        inner = getattr(model, '_orig_mod', model)
        inner.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_iter = ckpt["iter"] + 1
        best_val_loss = ckpt.get("best_val_loss", float('inf'))
        print(f"[Resume] Desde iter {start_iter} (best_val_loss={best_val_loss:.4f})")

    # Iterador infinito
    def infinite_loader(loader):
        while True:
            yield from loader

    train_iter = infinite_loader(train_loader)
    grad_accum = config["grad_accum"]

    print(f"\n[Train] {config['max_iters']} iters | batch efectivo {config['batch_size']*grad_accum}")
    print(f"[Train] {tokens_per_iter:,} tokens/iter | BF16 activado\n")

    t0 = time.time()
    running_loss = 0.0

    for it in range(start_iter, config["max_iters"] + 1):
        lr = get_lr(it)
        for group in optimizer.param_groups:
            group["lr"] = lr

        # --- Gradient accumulation con BF16 ---
        optimizer.zero_grad()
        accum_loss = 0.0

        for micro_step in range(grad_accum):
            x, y = next(train_iter)
            x, y = x.to(device), y.to(device)
            with autocast(device_type="cuda", dtype=torch.bfloat16):
                _, loss = model(x, y)
                loss = loss / grad_accum
            loss.backward()
            accum_loss += loss.item()

        torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
        optimizer.step()

        # Running average
        running_loss = accum_loss if running_loss == 0 else 0.99 * running_loss + 0.01 * accum_loss

        # --- Eval ---
        if it % config["eval_interval"] == 0 or it == start_iter:
            val_loss = evaluate(model, val_loader, device)
            dt = time.time() - t0
            t0 = time.time()
            print(f"  iter {it:6d} | train {running_loss:.4f} | val {val_loss:.4f} | "
                  f"lr {lr:.2e} | {dt:.1f}s")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                inner = getattr(model, '_orig_mod', model)
                torch.save({
                    "model_state": inner.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "config": config, "iter": it,
                    "val_loss": val_loss, "best_val_loss": best_val_loss,
                }, os.path.join(config["checkpoint_dir"], "best_model.pt"))
                print(f"           ↑ Mejor modelo (val_loss={val_loss:.4f})")

        # --- Checkpoint periódico ---
        if it % config["checkpoint_interval"] == 0:
            inner = getattr(model, '_orig_mod', model)
            torch.save({
                "model_state": inner.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "config": config, "iter": it,
                "val_loss": val_loss if it % config["eval_interval"] == 0 else None,
                "best_val_loss": best_val_loss,
            }, os.path.join(config["checkpoint_dir"], "latest.pt"))

    print(f"\n[Train] Completado. Mejor val_loss: {best_val_loss:.4f}")
    return model, tokenizer

In [ ]:
# --- Lanzar entrenamiento ---
model, tokenizer = train()

## 7. Generar texto

Temperatura más baja (0.3–0.5) da texto más conservador y factual.

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_tokens=200, temperature=0.5, top_k=20):
    device = next(model.parameters()).device
    model.eval()

    ids = tokenizer.encode(prompt)
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    raw_model = getattr(model, '_orig_mod', model)
    block_size = raw_model.block_size

    for _ in range(max_tokens):
        idx_cond = idx[:, -block_size:]
        with autocast(device_type="cuda", dtype=torch.bfloat16):
            logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < values[:, [-1]]] = float('-inf')
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)

    return tokenizer.decode(idx[0].tolist())

In [ ]:
# --- Probar generación ---
prompts = [
    "La inteligencia artificial",
    "La Segunda Guerra Mundial",
    "En la ciudad de Madrid",
    "La teoría de la relatividad",
    "El sistema solar",
]

for prompt in prompts:
    print(f"Prompt: '{prompt}'")
    print("─" * 60)
    print(generate(model, tokenizer, prompt, max_tokens=200, temperature=0.5, top_k=20))
    print("─" * 60)
    print()

## 8. Cargar modelo desde checkpoint

Si Colab se desconecta, ejecuta las celdas 1–4, luego esta para recuperar el modelo.

In [ ]:
!ls -lh "{PROJECT_DIR}/checkpoints/"

In [ ]:
# --- Descomentar para cargar modelo desde checkpoint ---

# device = "cuda" if torch.cuda.is_available() else "cpu"
#
# tokenizer = TiktokenWrapper()
# tokenizer.load(f"{PROJECT_DIR}/data/vocab.json")
#
# ckpt_path = f"{PROJECT_DIR}/checkpoints/best_model.pt"
# ckpt = torch.load(ckpt_path, map_location=device)
# cfg = ckpt["config"]
#
# model = GPT(
#     vocab_size=cfg["vocab_size"], n_embd=cfg["n_embd"],
#     n_head=cfg["n_head"], n_layer=cfg["n_layer"],
#     block_size=cfg["block_size"], dropout=cfg["dropout"],
# ).to(device)
# model.load_state_dict(ckpt["model_state"])
# model = torch.compile(model)
# print(f"Modelo cargado (iter {ckpt['iter']}, val_loss {ckpt['val_loss']:.4f})")